# HAR Batch Processor — one click from Drive to labeled sessions

**The workflow this enables:**
1. Record (app + glasses). App session -> share to Drive `HAR_data/sessions/`. Glasses video->you drop it in Drive `HAR_data/inbox_videos/` (or point Step 2 at your backup folder).
2. Open this notebook -> **Runtime -> Run all**.
3. It pairs each loose video with its session by start time, moves it into the session folder, runs the full labeling pipeline on every unprocessed session, and writes the outputs back to Drive.

No PC, no manual sorting, no per-session Colab dance. Already-processed sessions are skipped, so re-running is always safe.

**First-Time Setup:** create `HAR_data/` in your Google Drive with subfolders `sessions/` and `inbox_videos/`, and put `har_pipeline.py` inside `HAR_data/`.

In [ ]:
#1 — mount Drive + config
from google.colab import drive
drive.mount('/content/drive')

BASE         = "/content/drive/MyDrive/HAR_data"
SESSIONS_DIR = f"{BASE}/sessions"
INBOX_VIDEOS = f"{BASE}/inbox_videos"
PAIR_TOLERANCE_MIN = 5                   # max start-time gap for pairing

!pip -q install pymediainfo opencv-python-headless google-genai
import sys, os
sys.path.insert(0, BASE)                 # har_pipeline.py lives in HAR_data/
import har_pipeline as hp
os.makedirs(SESSIONS_DIR, exist_ok=True); os.makedirs(INBOX_VIDEOS, exist_ok=True)
print("ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ready


In [ ]:
import shutil, os
#Manual pairing. Disable if auto-pairing is active and properly working
PAIRS = {
    "small_mixed 1.mp4":    "mixed_01",
    "small_mixed 2.mp4":    "mixed_02",
    "small_mixed 5.mp4":    "mixed_03",
    "small_mixed 10.mp4":   "mixed_04",
    "small_other 1.mp4":    "other_01",
    "small_other 2.mp4":    "other_02",
    "small_running 1.mp4":  "running_01",
    "small_running 10.mp4": "running_02",
}

for vid, sess in PAIRS.items():
    src = os.path.join(INBOX_VIDEOS, vid)
    dst_dir = os.path.join(SESSIONS_DIR, sess)
    if not os.path.exists(src):
        print(f"missing video: {vid}"); continue
    if not os.path.isdir(dst_dir):
        print(f"missing session folder: {sess}"); continue
    shutil.move(src, os.path.join(dst_dir, "video.mp4"))
    print(f"paired {vid} -> {sess}/video.mp4")

paired small_mixed 1.mp4 -> mixed_01/video.mp4
paired small_mixed 2.mp4 -> mixed_02/video.mp4
paired small_mixed 5.mp4 -> mixed_03/video.mp4
paired small_mixed 10.mp4 -> mixed_04/video.mp4
paired small_other 1.mp4 -> other_01/video.mp4
paired small_other 2.mp4 -> other_02/video.mp4
paired small_running 1.mp4 -> running_01/video.mp4
paired small_running 10.mp4 -> running_02/video.mp4


## Step 1 — Pair loose videos with sessions
A session needs a video; videos arrive as loose files. Pairing key: the video's metadata start time vs the session's sensor start time. The glasses clock can be off by ~half a minute, but sessions are many minutes apart, so nearest-start within a tolerance is unambiguous — and anything ambiguous is flagged instead of guessed.

In [ ]:
#2 — auto-pairing
import glob, shutil, json
import pandas as pd

def session_start_ms(sdir):
    sj = os.path.join(sdir, "session.json")
    if os.path.exists(sj):
        with open(sj) as f:
            m = json.load(f)
        if m.get("recording_start_ms"): return int(m["recording_start_ms"])
    for n in ["sensors.csv", "sensor_data.csv"]:
        p = os.path.join(sdir, n)
        if os.path.exists(p):
            return int(pd.read_csv(p, nrows=1).Timestamp_ms.iloc[0])
    return None

def has_video(sdir):
    return any(p.lower().endswith((".mp4", ".mov"))
               for p in glob.glob(os.path.join(sdir, "*")))

sessions = [d for d in sorted(glob.glob(f"{SESSIONS_DIR}/*")) if os.path.isdir(d)]
need_video = [(d, session_start_ms(d)) for d in sessions if not has_video(d)]
need_video = [(d, s) for d, s in need_video if s is not None]
videos = sorted(glob.glob(f"{INBOX_VIDEOS}/*.mp4") + glob.glob(f"{INBOX_VIDEOS}/*.mov"))

print(f"{len(sessions)} sessions ({len(need_video)} need a video), "
      f"{len(videos)} videos in inbox")


tol_ms = PAIR_TOLERANCE_MIN * 60 * 1000
for vid in videos:
    v_start = hp.get_video_meta_start_ms(vid)
    if v_start is None:
        print(f"  SKIP {os.path.basename(vid)}: no metadata time"); continue
    cands = [(abs(s - v_start), d, s) for d, s in need_video if abs(s - v_start) < tol_ms]
    if not cands:
        print(f"  UNMATCHED {os.path.basename(vid)} — no session starts within "
              f"{PAIR_TOLERANCE_MIN} min"); continue
    cands.sort()
    if len(cands) > 1 and cands[1][0] < tol_ms:
        gap0, gap1 = cands[0][0]/1000, cands[1][0]/1000
        if gap1 - gap0 < 60_000/1000:
            print(f"  AMBIGUOUS {os.path.basename(vid)}: two sessions within a minute "
                  f"of each other — pair manually"); continue
    _, sdir, _ = cands[0]
    dest = os.path.join(sdir, "video" + os.path.splitext(vid)[1].lower())
    shutil.move(vid, dest)
    need_video = [(d, s) for d, s in need_video if d != sdir]
    print(f"  paired {os.path.basename(vid)} -> {os.path.basename(sdir)} "
          f"(start gap {cands[0][0]/1000:.0f}s)")

8 sessions (0 need a video), 0 videos in inbox


## Step 2 — Process every unprocessed session
One call per session; failures don't stop the batch. Outputs land inside each session folder on Drive: `synced_data.csv`, `windows_structured.csv`, `windows_labeled_final.csv`, plus the extracted frames.

In [ ]:
#3 — batch run
from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')

results = {}
for sdir in [d for d in sorted(glob.glob(f"{SESSIONS_DIR}/*")) if os.path.isdir(d)]:
    name = os.path.basename(sdir)
    print(f"\n=== {name}")
    try:
        if not has_video(sdir):
            print("  no video yet — skipping"); results[name] = "no video"; continue
        out = hp.process_session(sdir, api_key=API_KEY)
        results[name] = "done" if out is not None else "already done"
    except Exception as e:
        print(f"  FAILED: {e}")
        results[name] = f"FAILED: {e}"

print("\n================ summary")
for k, v in results.items():
    print(f"  {k:40s} {v}")


=== mixed_01
  video 300.1s, sensors 308.1s, trim tail 8.1s
  299 windows -> 3 segments -> 13 VLM chunks
  39 frames for 13 chunks
  labels: Desk work:153, Walking:124, Running:21, Stationary:1

=== mixed_02
  video 300.1s, sensors 186.0s, trim tail 0.0s
  184 windows -> 2 segments -> 8 VLM chunks
  24 frames for 8 chunks
  labels: Walking:141, Phone use:43

=== mixed_03
  video 180.0s, sensors 194.6s, trim tail 14.6s
  179 windows -> 10 segments -> 13 VLM chunks
  39 frames for 13 chunks
  labels: Walking:98, Running:47, Cooking:20, Phone use:9, Stationary:5

=== mixed_04
  video 180.0s, sensors 204.8s, trim tail 24.8s
  179 windows -> 2 segments -> 8 VLM chunks
  24 frames for 8 chunks
  labels: Walking:109, Phone use:69, Stationary:1

=== other_01
  video 180.0s, sensors 182.0s, trim tail 2.0s
  179 windows -> 1 segments -> 8 VLM chunks
  24 frames for 8 chunks
  labels: Phone use:179

=== other_02
  video 180.0s, sensors 182.1s, trim tail 2.0s
  179 windows -> 1 segments -> 8 VLM 

## Step 3 — Dataset overview
Everything labeled so far, in one table — this doubles as the recording to-do list (classes with few sessions need more).

In [ ]:
#4 — overview across all sessions
frames = []
for sdir in sorted(glob.glob(f"{SESSIONS_DIR}/*")):
    p = os.path.join(sdir, "windows_labeled_final.csv")
    if os.path.exists(p):
        df = pd.read_csv(p); df["session"] = os.path.basename(sdir)
        frames.append(df)
if frames:
    all_w = pd.concat(frames)
    print(f"{all_w.session.nunique()} labeled sessions, {len(all_w)} windows\n")
    print(all_w.groupby("Label")["session"].agg(windows="count", sessions="nunique"))
else:
    print("no labeled sessions yet")

8 labeled sessions, 1556 windows

            windows  sessions
Label                        
Cooking          20         1
Desk work       153         1
Phone use       479         5
Running         247         4
Stationary       49         4
Walking         608         6
